<a href="https://colab.research.google.com/github/sokrypton/7.571/blob/main/L8/hierarchical_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌳 Hierarchical Clustering

Same algorithm, one difference — how do you measure distance between clusters?

| Method | Distance between clusters | Tendency |
|---|---|---|
| **Single Linkage** | Minimum (closest pair) | Chains — elongated clusters |
| **Complete Linkage** | Maximum (farthest pair) | Compact, round clusters |
| **Average Linkage (UPGMA)** | Mean of all pairs | Middle ground |

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

## 1. When Does the Linkage Method Matter?

With well-separated round blobs, all methods agree.  
The differences show up when clusters are **non-spherical**.

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=150, centers=3, cluster_std=1.0, random_state=42)
X_moons, y_moons = make_moons(n_samples=200, noise=0.05, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='Set1', s=20)
axes[0].set_title('Blobs (easy)', fontweight='bold')
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='Set1', s=20)
axes[1].set_title('Moons (tricky)', fontweight='bold'); axes[1].set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Blobs — all three methods agree
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, method in zip(axes, ['single', 'complete', 'average']):
    clusters = AgglomerativeClustering(n_clusters=3, linkage=method).fit_predict(X_blobs)
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=clusters, cmap='Set1', s=20)
    ax.set_title(f'{method.capitalize()}', fontsize=13, fontweight='bold')
plt.suptitle('Blobs — All Methods Agree', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Moons — now they disagree!
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, method in zip(axes, ['single', 'complete', 'average']):
    clusters = AgglomerativeClustering(n_clusters=2, linkage=method).fit_predict(X_moons)
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=clusters, cmap='Set1', s=15)
    ax.set_title(f'{method.capitalize()}', fontsize=13, fontweight='bold')
    ax.set_aspect('equal')
plt.suptitle('Moons — Methods Disagree!', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()
print('👆 Single linkage chains along each curve — correct!')
print('   Complete and average use global distances — wrong for this shape.')

---
## 2. Dendrograms

The full tree shows every merge. `n_clusters` in sklearn just cuts this tree early —  
it's the same bottom-up algorithm as UPGMA, you just choose where to stop.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, method in zip(axes, ['single', 'complete', 'average']):
    Z = linkage(X_moons, method=method)
    dendrogram(Z, ax=ax, leaf_rotation=90, no_labels=True)
    ax.set_title(f'{method.capitalize()} Linkage', fontsize=14, fontweight='bold')
    ax.set_ylabel('Distance')
plt.tight_layout()
plt.show()

---
## 3. Real Data — Animal Phylogeny

Distance matrix from immunological comparison of albumin proteins.  
Average linkage = UPGMA — the method you already know from lecture.

In [ ]:
dm = np.array(
    [[  0, 32, 48, 51, 50, 48, 98,148],
     [ 32,  0, 26, 34, 29, 33, 84,136],
     [ 48, 26,  0, 42, 44, 44, 92,152],
     [ 51, 34, 42,  0, 44, 38, 86,142],
     [ 50, 29, 44, 44,  0, 24, 89,142],
     [ 48, 33, 44, 38, 24,  0, 90,142],
     [ 98, 84, 92, 86, 89, 90,  0,148],
     [148,136,152,142,142,142,148,  0]], dtype=float)

labels = ['Dog', 'Bear', 'Raccoon', 'Weasel', 'Seal', 'SeaLion', 'Cat', 'Monkey']
dm_condensed = squareform(dm)  # scipy needs upper triangle form

plt.figure(figsize=(6, 5))
plt.imshow(dm, cmap='Greys')
plt.xticks(range(8), labels, rotation=45, ha='right')
plt.yticks(range(8), labels)
plt.colorbar(label='Distance')
plt.title('Pairwise Distance Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, method in zip(axes, ['single', 'complete', 'average']):
    Z = linkage(dm_condensed, method=method)
    dendrogram(Z, labels=labels, ax=ax, leaf_rotation=45, leaf_font_size=10)
    ax.set_title(f'{method.capitalize()} Linkage', fontsize=14, fontweight='bold')
    ax.set_ylabel('Distance')
plt.tight_layout()
plt.show()

---
## 4. Bonus: UPGMA from Scratch

The core algorithm — the **only** line that changes between methods  
is how you compute the distance to the new merged cluster.

In [ ]:
def hierarchical_clustering(dm, labels, method='average'):
    """
    Agglomerative clustering from scratch.
    method: 'single' (min), 'complete' (max), 'average' (UPGMA)
    """
    dm = np.array(dm, dtype=float)
    labels = list(labels)
    sizes = {lab: 1 for lab in labels}
    history = []

    while len(labels) > 1:
        n = len(labels)

        # Find closest pair
        min_dist, mi, mj = np.inf, 0, 1
        for i in range(n):
            for j in range(i + 1, n):
                if dm[i, j] < min_dist:
                    min_dist, mi, mj = dm[i, j], i, j

        history.append((labels[mi], labels[mj], min_dist))
        new_label = f'({labels[mi]}+{labels[mj]})'

        # Distance from new cluster to every other node
        ni, nj = sizes[labels[mi]], sizes[labels[mj]]
        new_row = []
        for k in range(n):
            if k in (mi, mj):
                continue
            ########################################
            # THIS IS THE ONLY LINE THAT CHANGES! #
            ########################################
            if method == 'single':    d = min(dm[mi, k], dm[mj, k])
            elif method == 'complete': d = max(dm[mi, k], dm[mj, k])
            else:                      d = (dm[mi, k] * ni + dm[mj, k] * nj) / (ni + nj)
            new_row.append(d)

        # Remove merged rows/cols, add new one
        keep = [k for k in range(n) if k not in (mi, mj)]
        dm = dm[np.ix_(keep, keep)]
        new_row = np.array(new_row)
        dm = np.vstack([dm, new_row[None, :]])
        dm = np.hstack([dm, np.append(new_row, 0)[:, None]])

        sizes[new_label] = ni + nj
        labels = [labels[k] for k in keep] + [new_label]

    return history


for method in ['single', 'complete', 'average']:
    print(f'\n--- {method.upper()} ---')
    for i, (a, b, d) in enumerate(hierarchical_clustering(dm, labels, method)):
        print(f'  Step {i+1}: merge {a} + {b}  (dist={d:.1f})')

---
## 📝 Key Takeaways

| Concept | Detail |
|---|---|
| **Algorithm** | Find closest pair → merge → update distances → repeat |
| **Single** | min distance — chains, finds irregular shapes |
| **Complete** | max distance — compact, round clusters |
| **Average (UPGMA)** | mean distance — balanced, used in phylogenetics |
| **vs K-Means** | No need to choose K upfront — cut the dendrogram anywhere |
| **sklearn** | `AgglomerativeClustering(n_clusters=K, linkage=method)` |